In [ ]:
import os

repo_name = 'deepl-cifar100-image-analysis'
if not os.path.exists(repo_name):
    get_ipython().system(f'git clone https://github.com/Fgram-devAI/{repo_name}.git')
else:
    print(f'Repository "{repo_name}" already exists. Skipping clone.')

In [ ]:
%cd /content/deepl-cifar100-image-analysis

In [ ]:
get_ipython().system('git checkout -b experiment/efficientnet-b0-fine')
get_ipython().system('pwd')

# Verify Python Libraries

In [ ]:
import sklearn, numpy
print("sklearn:", sklearn.__version__)
print("numpy:", numpy.__version__)


# EfficientNetB0 Model Definition

In [ ]:
# %%writefile models/efficientnet_b0_fine.py
"""EfficientNetB0 transfer-learning model for CIFAR-100 multiclass tasks."""

import tensorflow as tf

from models.augmentation import build_augmentation

keras = tf.keras
layers = tf.keras.layers

# EfficientNetB0 backbone has 237 layers total, grouped into 7 MBConv blocks
# (block1..block7) plus a stem and a top. Layer-index where each block starts
# (measured at input_size=128; stable across input sizes since it's topology,
# not shape, that determines block boundaries):
#   stem -> 3   block1 -> 7   block2 -> 17   block3 -> 46   block4 -> 75
#   block5 -> 119   block6 -> 162   block7 -> 221   top -> 234   (237 total)
# So unfreeze_from="block6" leaves the first ~68% of the network frozen and
# trains only the last ~75 layers (block6, block7, top) — usually the best
# accuracy/stability tradeoff for a single-stage fine-tune.


def build_efficientnet_b0(
    input_shape: tuple[int, int, int] = (32, 32, 3),
    num_classes: int = 100,
    dropout: float = 0.4,
    freeze_backbone: bool = True,
    unfreeze_from: str | None = None,   # None | "all" | "block5" | "block6" | "block7" | "top"
    freeze_bn: bool = True,             # keep BatchNorm stats frozen (recommended)
    input_size: int = 128,
    augmentation: dict | None = None,
) -> keras.Model:
    """EfficientNetB0 classifier for CIFAR-100.

    32x32 -> resize(input_size) -> rescale to [0,255] (EfficientNet has its own
    internal normalization) -> ImageNet backbone -> dropout + softmax head.
    Returned uncompiled.

    Backbone freezing modes:
      - freeze_backbone=True: everything frozen (fastest, weakest).
      - freeze_backbone=False, unfreeze_from=None or "all": everything
        trainable (strongest ceiling, but riskiest single-stage — needs a
        very low learning_rate, e.g. 1e-5).
      - freeze_backbone=False, unfreeze_from="block6" (etc.): only layers
        from that block onward are trainable; everything before it stays
        frozen. This is the recommended middle ground — adapts the
        high-level semantic features without disturbing low-level filters,
        and tolerates a higher learning_rate (~1e-4).

    freeze_bn=True additionally keeps every BatchNormalization layer frozen
    (non-trainable + inference-mode stats) even when its block is unfrozen.
    This avoids corrupting running mean/variance on small batches — almost
    always what you want when fine-tuning a pretrained backbone.
    """
    if num_classes < 1:
        raise ValueError(f"num_classes must be >= 1; got {num_classes}")

    inputs = keras.Input(shape=input_shape, name="image")

    aug_layer = build_augmentation(augmentation)
    x = aug_layer(inputs) if aug_layer is not None else inputs

    x = layers.Resizing(input_size, input_size, interpolation="bilinear")(x)
    x = layers.Rescaling(255.0)(x)

    backbone = keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(input_size, input_size, 3),
        pooling="avg",
    )

    if freeze_backbone:
        backbone.trainable = False
    else:
        backbone.trainable = True
        if unfreeze_from not in (None, "all"):
            # Freeze everything before `unfreeze_from`; train from there on.
            trainable = False
            for layer in backbone.layers:
                if layer.name.startswith(unfreeze_from):
                    trainable = True
                layer.trainable = trainable
        if freeze_bn:
            for layer in backbone.layers:
                if isinstance(layer, layers.BatchNormalization):
                    layer.trainable = False

    # Run in inference mode (training=False) whenever the backbone is fully
    # frozen, so BN uses its pretrained running stats rather than batch stats.
    backbone_is_fully_frozen = freeze_backbone
    x = backbone(x, training=not backbone_is_fully_frozen)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="prob")(x)

    return keras.Model(inputs=inputs, outputs=outputs, name="efficientnet_b0")

# Register Model in `__init__.py`

In [ ]:
# %% ===== CELL 6: register the builder in models/__init__.py =====
%%writefile models/__init__.py
"""Models module: sequential, ViT, transfer-learning, and baseline CNN architectures."""

from .baseline import build_baseline_cnn
from .efficientnet_b0 import build_efficientnet_b0
from .sequential import build_rnn, build_lstm, build_bilstm
from .vit import build_vit
from .transfer import build_transfer

__all__ = [
    "build_baseline_cnn",
    "build_efficientnet_b0",
    "build_rnn",
    "build_lstm",
    "build_bilstm",
    "build_vit",
    "build_transfer",
]


# Patch `training/train.py` for EfficientNetB0

In [ ]:

import pathlib

p = pathlib.Path("training/train.py")
src = p.read_text()

if "from models.efficientnet_b0 import build_efficientnet_b0" not in src:
    src = src.replace(
        "from models.baseline import build_baseline_cnn\n",
        "from models.baseline import build_baseline_cnn\n"
        "from models.efficientnet_b0 import build_efficientnet_b0\n",
    )

old_build_model = '''def _build_model(config: Dict[str, Any], *, num_classes: int = 1) -> tf.keras.Model:
    architecture = config.get("architecture", "baseline_cnn")
    if architecture != "baseline_cnn":
        raise ValueError(
            f"This branch only ships the 'baseline_cnn' architecture; "
            f"got {architecture!r}"
        )
    return build_baseline_cnn(
        dropout=float(config.get("dropout", 0.3)),
        num_classes=num_classes,
        augmentation=config.get("augmentation"),
    )'''

new_build_model = '''def _build_model(config: Dict[str, Any], *, num_classes: int = 1) -> tf.keras.Model:
    architecture = config.get("architecture", "baseline_cnn")
    if architecture == "baseline_cnn":
        return build_baseline_cnn(
            dropout=float(config.get("dropout", 0.3)),
            num_classes=num_classes,
            augmentation=config.get("augmentation"),
        )
    if architecture == "efficientnet_b0":
        return build_efficientnet_b0(
            dropout=float(config.get("dropout", 0.3)),
            num_classes=num_classes,
            freeze_backbone=bool(config.get("freeze_backbone", True)),
            unfreeze_from=config.get("unfreeze_from"),
            freeze_bn=bool(config.get("freeze_bn", True)),
            input_size=int(config.get("input_size", 96)),
            augmentation=config.get("augmentation"),
        )
    raise ValueError(
        f"Unsupported architecture {architecture!r}. "
        "Supported architectures are 'baseline_cnn' and 'efficientnet_b0'."
    )'''

if "unfreeze_from" not in src:
    assert old_build_model in src, "train.py upstream changed — patch needs review"
    src = src.replace(old_build_model, new_build_model)
    p.write_text(src)
    print("train.py patched (now supports unfreeze_from / freeze_bn).")
else:
    print("train.py already supports unfreeze_from / freeze_bn.")


# Define Config: EfficientNetB0 Frozen Backbone

In [ ]:
import yaml

config_str = """
architecture: efficientnet_b0
run_name: efficientnet_b0_fine_frozen_in128_ep30
seed: 42

task:
  type: multiclass
  label_level: fine          # <-- 100 classes (was: coarse / 20)

validation:
  fraction: 0.1

class_imbalance:
  strategy: none             # CIFAR-100 is balanced (500/class)

batch_size: 64
shuffle_buffer: 4096
dropout: 0.4                 # a touch higher for the harder 100-class task
epochs: 30                  # was 10
optimizer: adam
learning_rate: 1e-3         # ok while the backbone is FROZEN

freeze_backbone: true       # this config keeps it frozen (the "better baseline")
input_size: 128             # was 96; try 160 if the GPU allows (drop batch_size to 32)

augmentation:               # exact keys expected by models/augmentation.py
  enabled: true             # REQUIRED — without this the whole block is ignored
  horizontal_flip: true
  translation: 0.1
  zoom: 0.1
  rotation: 0.05
  contrast: 0.1

early_stopping:
  monitor: val_loss
  patience: 8               # was 5

output_dir: results/efficientnet_b0_fine_frozen
save_weights: true
subset_size: null
"""

config = yaml.safe_load(config_str)
print(config)

# Verify Model and Config Setup

In [ ]:

get_ipython().system('grep -n efficientnet_b0 models/__init__.py')
get_ipython().system('grep -n "unfreeze_from\\|freeze_bn" training/train.py')
get_ipython().system('ls configs/multiclass/ | grep fine')


# Install Keras and Set Environment Variable

In [ ]:
get_ipython().system('pip install -q tf_keras')
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"


# Train Model: EfficientNetB0 Frozen Backbone (Run 1)

In [ ]:
import os

# Get the current working directory, which is the repository root.
# This directory needs to be in the PYTHONPATH for 'python -m' to find 'models'.
current_dir = os.getcwd()

# Construct the PYTHONPATH string, ensuring the current directory is first
# and preserving any existing PYTHONPATH entries. Using os.pathsep for cross-platform compatibility.
python_path_value = f"{current_dir}{os.pathsep}{os.environ.get('PYTHONPATH', '')}"

# Construct the full shell command, explicitly setting PYTHONPATH for the subprocess.
# This ensures the environment variable is correctly passed.
command = (
    f'PYTHONPATH="{python_path_value}" '
    'python -m training.train --config configs/multiclass/efficientnet_b0_fine_frozen.yaml'
)

get_ipython().system(command)

# Write EfficientNetB0 Model File

In [ ]:
import pathlib

# Get the content of cell Ckq0457eNCTG
efficientnet_b0_code = '''
"""EfficientNetB0 transfer-learning model for CIFAR-100 multiclass tasks."""

import tensorflow as tf

from models.augmentation import build_augmentation

keras = tf.keras
layers = tf.keras.layers

# EfficientNetB0 backbone has 237 layers total, grouped into 7 MBConv blocks
# (block1..block7) plus a stem and a top. Layer-index where each block starts
# (measured at input_size=128; stable across input sizes since it's topology,
# not shape, that determines block boundaries):
#   stem -> 3   block1 -> 7   block2 -> 17   block3 -> 46   block4 -> 75
#   block5 -> 119   block6 -> 162   block7 -> 221   top -> 234   (237 total)
# So unfreeze_from="block6" leaves the first ~68% of the network frozen and
# trains only the last ~75 layers (block6, block7, top) — usually the best
# accuracy/stability tradeoff for a single-stage fine-tune.


def build_efficientnet_b0(
    input_shape: tuple[int, int, int] = (32, 32, 3),
    num_classes: int = 100,
    dropout: float = 0.4,
    freeze_backbone: bool = True,
    unfreeze_from: str | None = None,   # None | "all" | "block5" | "block6" | "block7" | "top"
    freeze_bn: bool = True,             # keep BatchNorm stats frozen (recommended)
    input_size: int = 128,
    augmentation: dict | None = None,
) -> keras.Model:
    """EfficientNetB0 classifier for CIFAR-100.

    32x32 -> resize(input_size) -> rescale to [0,255] (EfficientNet has its own
    internal normalization) -> ImageNet backbone -> dropout + softmax head.
    Returned uncompiled.

    Backbone freezing modes:
      - freeze_backbone=True: everything frozen (fastest, weakest).
      - freeze_backbone=False, unfreeze_from=None or "all": everything
        trainable (strongest ceiling, but riskiest single-stage — needs a
        very low learning_rate, e.g. 1e-5).
      - freeze_backbone=False, unfreeze_from="block6" (etc.): only layers
        from that block onward are trainable; everything before it stays
        frozen. This is the recommended middle ground — adapts the
        high-level semantic features without disturbing low-level filters,
        and tolerates a higher learning_rate (~1e-4).

    freeze_bn=True additionally keeps every BatchNormalization layer frozen
    (non-trainable + inference-mode stats) even when its block is unfrozen.
    This avoids corrupting running mean/variance on small batches — almost
    always what you want when fine-tuning a pretrained backbone.
    """
    if num_classes < 1:
        raise ValueError(f"num_classes must be >= 1; got {num_classes}")

    inputs = keras.Input(shape=input_shape, name="image")

    aug_layer = build_augmentation(augmentation)
    x = aug_layer(inputs) if aug_layer is not None else inputs

    x = layers.Resizing(input_size, input_size, interpolation="bilinear")(x)
    x = layers.Rescaling(255.0)(x)

    backbone = keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(input_size, input_size, 3),
        pooling="avg",
    )

    if freeze_backbone:
        backbone.trainable = False
    else:
        backbone.trainable = True
        if unfreeze_from not in (None, "all"):
            # Freeze everything before `unfreeze_from`; train from there on.
            trainable = False
            for layer in backbone.layers:
                if layer.name.startswith(unfreeze_from):
                    trainable = True
                layer.trainable = trainable
        if freeze_bn:
            for layer in backbone.layers:
                if isinstance(layer, layers.BatchNormalization):
                    layer.trainable = False

    # Run in inference mode (training=False) whenever the backbone is fully
    # frozen, so BN uses its pretrained running stats rather than batch stats.
    backbone_is_fully_frozen = freeze_backbone
    x = backbone(x, training=not backbone_is_fully_frozen)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="prob")(x)

    return keras.Model(inputs=inputs, outputs=outputs, name="efficientnet_b0")
'''

# Define the path where the file should be created
file_path = pathlib.Path('models/efficientnet_b0.py')

# Create the file and write the code to it
file_path.write_text(efficientnet_b0_code)

print(f'Created {file_path} with the EfficientNetB0 model definition.')


# Write Config File: EfficientNetB0 Frozen Backbone

In [ ]:
# %%writefile configs/multiclass/efficientnet_b0_fine_frozen.yaml
import pathlib

# Get the config string from the kernel state (it was previously loaded in H2aIfhr7Nn2k)
# For robustness, we will redefine it here to ensure it's available.
config_str_for_file = '''
architecture: efficientnet_b0
run_name: efficientnet_b0_fine_frozen_in128_ep30
seed: 42

task:
  type: multiclass
  label_level: fine          # <-- 100 classes (was: coarse / 20)

validation:
  fraction: 0.1

class_imbalance:
  strategy: none             # CIFAR-100 is balanced (500/class)

batch_size: 64
shuffle_buffer: 4096
dropout: 0.4                 # a touch higher for the harder 100-class task
epochs: 30                  # was 10
optimizer: adam
learning_rate: 1e-3         # ok while the backbone is FROZEN

freeze_backbone: true       # this config keeps it frozen (the "better baseline")
input_size: 128             # was 96; try 160 if the GPU allows (drop batch_size to 32)

augmentation:               # exact keys expected by models/augmentation.py
  enabled: true             # REQUIRED — without this the whole block is ignored
  horizontal_flip: true
  translation: 0.1
  zoom: 0.1
  rotation: 0.05
  contrast: 0.1

early_stopping:
  monitor: val_loss
  patience: 8               # was 5

output_dir: results/efficientnet_b0_fine_frozen
save_weights: true
subset_size: null
'''

# Define the path where the config file should be created
config_file_path = pathlib.Path('configs/multiclass/efficientnet_b0_fine_frozen.yaml')

# Create the necessary directories if they don't exist
config_file_path.parent.mkdir(parents=True, exist_ok=True)

# Write the config string to the file
config_file_path.write_text(config_str_for_file)

print(f'Created config file: {config_file_path}')


# Train Model: EfficientNetB0 Frozen Backbone

In [ ]:
import os

# Get the current working directory, which is the repository root.
# This directory needs to be in the PYTHONPATH for 'python -m' to find 'models'.
current_dir = os.getcwd()

# Construct the PYTHONPATH string, ensuring the current directory is first
# and preserving any existing PYTHONPATH entries. Using os.pathsep for cross-platform compatibility.
python_path_value = f"{current_dir}{os.pathsep}{os.environ.get('PYTHONPATH', '')}"

# Construct the full shell command, explicitly setting PYTHONPATH for the subprocess.
# This ensures the environment variable is correctly passed.
command = (
    f'PYTHONPATH="{python_path_value}" '
    'python -m training.train --config configs/multiclass/efficientnet_b0_fine_frozen.yaml'
)

get_ipython().system(command)


# Attempt Train Model: EfficientNetB0 Unfreeze Block6 (Error)

In [ ]:
get_ipython().system('python -m training.train --config configs/multiclass/efficientnet_b0_fine_unfreeze_block6.yaml')

# Write Config File: EfficientNetB0 Unfreeze Block6

In [ ]:
# %%writefile configs/multiclass/efficientnet_b0_fine_unfreeze_block6.yaml
import pathlib

config_str_unfreeze = '''
architecture: efficientnet_b0
run_name: efficientnet_b0_fine_unfreeze_block6_in128_ep30
seed: 42

task:
  type: multiclass
  label_level: fine          # <-- 100 classes (was: coarse / 20)

validation:
  fraction: 0.1

class_imbalance:
  strategy: none             # CIFAR-100 is balanced (500/class)

batch_size: 64
shuffle_buffer: 4096
dropout: 0.4                 # a touch higher for the harder 100-class task
epochs: 30                  # was 10
optimizer: adam
learning_rate: 1e-4         # lower learning rate needed when unfreezing more layers

freeze_backbone: false      # Now we are unfreezing parts of the backbone
unfreeze_from: block6       # unfreeze from block6 onwards
input_size: 128             # was 96; try 160 if the GPU allows (drop batch_size to 32)

augmentation:               # exact keys expected by models/augmentation.py
  enabled: true             # REQUIRED — without this the whole block is ignored
  horizontal_flip: true
  translation: 0.1
  zoom: 0.1
  rotation: 0.05
  contrast: 0.1

early_stopping:
  monitor: val_loss
  patience: 8               # was 5

output_dir: results/efficientnet_b0_fine_unfreeze_block6
save_weights: true
subset_size: null
'''

# Define the path where the new config file should be created
config_file_path_unfreeze = pathlib.Path('configs/multiclass/efficientnet_b0_fine_unfreeze_block6.yaml')

# Create the necessary directories if they don't exist
config_file_path_unfreeze.parent.mkdir(parents=True, exist_ok=True)

# Write the config string to the file
config_file_path_unfreeze.write_text(config_str_unfreeze)

print(f'Created config file: {config_file_path_unfreeze}')


# Train Model: EfficientNetB0 Unfreeze Block6 (Run 2)

In [ ]:
import os

# Get the current working directory, which is the repository root.
# This directory needs to be in the PYTHONPATH for 'python -m' to find 'models'.
current_dir = os.getcwd()

# Construct the PYTHONPATH string, ensuring the current directory is first
# and preserving any existing PYTHONPATH entries. Using os.pathsep for cross-platform compatibility.
python_path_value = f"{current_dir}{os.pathsep}{os.environ.get('PYTHONPATH', '')}"

# Construct the full shell command, explicitly setting PYTHONPATH for the subprocess.
# This ensures the environment variable is correctly passed.
command = (
    f'PYTHONPATH="{python_path_value}" '
    'python -m training.train --config configs/multiclass/efficientnet_b0_fine_unfreeze_block6.yaml'
)

get_ipython().system(command)


# Write Config File: EfficientNetB0 Full Fine-tune

In [ ]:
# %%writefile configs/multiclass/efficientnet_b0_fine_ft.yaml
import pathlib
import yaml

config_str_ft = '''
architecture: efficientnet_b0
run_name: efficientnet_b0_fine_ft_in128_ep40
snapshot_weights: results/efficientnet_b0_fine_unfreeze_block6/efficientnet_b0_fine_unfreeze_block6_in128_ep30/weights.h5
seed: 42

task:
  type: multiclass
  label_level: fine

validation:
  fraction: 0.1

class_imbalance:
  strategy: none

batch_size: 64
shuffle_buffer: 4096
dropout: 0.4
epochs: 40
optimizer: adam
learning_rate: 1e-5

freeze_backbone: false
unfreeze_from: all
freeze_bn: true
input_size: 128

augmentation:
  enabled: true
  horizontal_flip: true
  translation: 0.1
  zoom: 0.1
  rotation: 0.05
  contrast: 0.1

early_stopping:
  monitor: val_loss
  patience: 10

output_dir: results/efficientnet_b0_fine_ft
save_weights: true
subset_size: null
'''

# Define the path where the new config file should be created
config_file_path_ft = pathlib.Path('configs/multiclass/efficientnet_b0_fine_ft.yaml')

# Create the necessary directories if they don't exist
config_file_path_ft.parent.mkdir(parents=True, exist_ok=True)

# Write the config string to the file
config_file_path_ft.write_text(config_str_ft)

print(f'Created config file: {config_file_path_ft}')


# Train Model: EfficientNetB0 Full Fine-tune (Run 3)

In [ ]:
get_ipython().system('python -m training.train --config configs/multiclass/efficientnet_b0_fine_ft.yaml')


# Verify All Three Runs Output

In [ ]:
# %% ===== CELL CHECK: verify all three runs have complete outputs =====
import os

for run_path in [
    "results/efficientnet_b0_fine_frozen",
    "results/efficientnet_b0_fine_unfreeze_block6",
    "results/efficientnet_b0_fine_ft",
]:
    print(f"\n=== {run_path} ===")
    if not os.path.isdir(run_path):
        print("  MISSING — this run folder doesn't exist")
        continue
    for root, dirs, files in os.walk(run_path):
        for f in files:
            full = os.path.join(root, f)
            size = os.path.getsize(full)
            print(f"  {full}  ({size:,} bytes)")

# List Contents of Run Directory

In [ ]:
from pathlib import Path
run_dir = Path("results/efficientnet_b0_fine_unfreeze_block6/efficientnet_b0_fine_unfreeze_block6_in128_ep30")
get_ipython().system(f'ls -la {run_dir}')

# Load History and Metrics

In [ ]:
import json

with open(run_dir / "history.json") as f:
    history = json.load(f)
with open(run_dir / "metrics.json") as f:
    metrics = json.load(f)

print("History keys:", list(history.keys()))
print("=== Final test metrics (fine / 100 classes) ===")
print(f"Accuracy:        {metrics['accuracy']:.4f}")
if "top_3_accuracy" in metrics:
    print(f"Top-3 accuracy:  {metrics['top_3_accuracy']:.4f}")
if "top_5_accuracy" in metrics:
    print(f"Top-5 accuracy:  {metrics['top_5_accuracy']:.4f}")
print(f"Macro precision: {metrics['macro_precision']:.4f}")
print(f"Macro recall:    {metrics['macro_recall']:.4f}")
print(f"Macro F1:        {metrics['macro_f1']:.4f}")
print(f"Weighted F1:     {metrics['weighted_f1']:.4f}")

# Append Metrics to Summary CSV

In [ ]:
import csv

summary_path = Path("results/summary.csv")
COLUMNS = ["task_name", "run_name", "task_type", "label_level", "target_labels",
           "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "threshold",
           "top_3_accuracy", "top_5_accuracy", "macro_f1", "weighted_f1"]

row = {c: "" for c in COLUMNS}
row.update({
    "task_name": run_dir.parent.name,     # e.g. efficientnet_b0_fine_ft
    "run_name": run_dir.name,
    "task_type": "multiclass",
    "label_level": "fine",
    "accuracy": metrics.get("accuracy", ""),
    "top_3_accuracy": metrics.get("top_3_accuracy", ""),
    "top_5_accuracy": metrics.get("top_5_accuracy", ""),
    "macro_f1": metrics.get("macro_f1", ""),
    "weighted_f1": metrics.get("weighted_f1", ""),
})

write_header = not summary_path.exists()
with summary_path.open("a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=COLUMNS)
    if write_header:
        w.writeheader()
    w.writerow(row)
print(f"Appended {row['run_name']} to {summary_path}")


# Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(history["accuracy"], label="train")
axes[1].plot(history["val_accuracy"], label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout()
plt.savefig(run_dir / "training_curves.png", dpi=150)
plt.show()


# Define Fine-Grained Labels

In [ ]:
FINE_LABELS = [
    "apple", "aquarium_fish", "baby", "bear", "beaver", "bed", "bee", "beetle",
    "bicycle", "bottle", "bowl", "boy", "bridge", "bus", "butterfly", "camel",
    "can", "castle", "caterpillar", "cattle", "chair", "chimpanzee", "clock",
    "cloud", "cockroach", "couch", "crab", "crocodile", "cup", "dinosaur",
    "dolphin", "elephant", "flatfish", "forest", "fox", "girl", "hamster",
    "house", "kangaroo", "keyboard", "lamp", "lawn_mower", "leopard", "lion",
    "lizard", "lobster", "man", "maple_tree", "motorcycle", "mountain", "mouse",
    "mushroom", "oak_tree", "orange", "orchid", "otter", "palm_tree", "pear",
    "pickup_truck", "pine_tree", "plain", "plate", "poppy", "porcupine",
    "possum", "rabbit", "raccoon", "ray", "road", "rocket", "rose", "sea",
    "seal", "shark", "shrew", "skunk", "skyscraper", "snail", "snake", "spider",
    "squirrel", "streetcar", "sunflower", "sweet_pepper", "table", "tank",
    "telephone", "television", "tiger", "tractor", "train", "trout", "tulip",
    "turtle", "wardrobe", "whale", "willow_tree", "wolf", "woman", "worm",
]

assert len(FINE_LABELS) == 100


# Plot Confusion Matrix and Worst-20 F1

In [ ]:
import numpy as np

cm = np.array(metrics["confusion_matrix"])
fig, ax = plt.subplots(figsize=(11, 10))
im = ax.imshow(cm, cmap="Blues")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion matrix — fine (100 classes)")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(run_dir / "confusion_matrix.png", dpi=150)
plt.show()

# %% ===== CELL 18: per-class F1 — show the 20 WORST classes (100 bars is unreadable) =====
diag = np.diag(cm)
col_sums = cm.sum(axis=0); row_sums = cm.sum(axis=1)
precision = np.divide(diag, col_sums, out=np.zeros_like(diag, float), where=col_sums != 0)
recall    = np.divide(diag, row_sums, out=np.zeros_like(diag, float), where=row_sums != 0)
f1 = np.divide(2 * precision * recall, precision + recall,
               out=np.zeros_like(diag, float), where=(precision + recall) != 0)

worst = np.argsort(f1)[:20]   # 20 weakest classes
fig, ax = plt.subplots(figsize=(10, 8))
y = np.arange(len(worst))
ax.barh(y, f1[worst], color="#5DCAA5")
ax.set_yticks(y); ax.set_yticklabels([FINE_LABELS[i] for i in worst])
ax.set_xlabel("F1 score"); ax.set_title("20 worst fine classes by F1")
plt.tight_layout()
plt.savefig(run_dir / "per_class_f1_worst20.png", dpi=150)
plt.show()

# Visualize Misclassified Examples for 10 Worst Classes

In [ ]:
# %% ===== CELL 20: predict + per-category inspection =====
test_images_normalized = test_images.astype(np.float32) / 255.0
probs = model.predict(test_images_normalized, batch_size=64, verbose=1)
preds = probs.argmax(axis=1)
confidences = probs.max(axis=1)

CATEGORY = "trout"   # <- change to any of the 100 fine class names
cat_id = FINE_LABELS.index(CATEGORY)

is_this_class = test_labels == cat_id
correct_idx = np.flatnonzero(is_this_class & (preds == cat_id))[:5]
wrong_idx   = np.flatnonzero(is_this_class & (preds != cat_id))[:5]
print(f"'{CATEGORY}': {len(correct_idx)} correct shown, {len(wrong_idx)} wrong shown "
      f"(total {is_this_class.sum()}, acc {(preds[is_this_class] == cat_id).mean():.2%})")

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, idx in enumerate(correct_idx):
    axes[0, i].imshow(test_images[idx])
    axes[0, i].set_title(f"OK {FINE_LABELS[preds[idx]]}\n({confidences[idx]:.0%})", fontsize=8, color="green")
    axes[0, i].axis("off")
for i, idx in enumerate(wrong_idx):
    axes[1, i].imshow(test_images[idx])
    axes[1, i].set_title(f"X pred: {FINE_LABELS[preds[idx]]}\n({confidences[idx]:.0%})", fontsize=8, color="red")
    axes[1, i].axis("off")
fig.suptitle(f"Category: {CATEGORY} (true label for every image)", fontsize=12)
plt.tight_layout()
plt.savefig(Path(run_dir).parent / f"examples_{CATEGORY}.png", dpi=150)
plt.show()Show more1:58 μ.μ.Claude responded: ΒΒClaude's response was interrupted.Try againYou said: καλο θα ηταν αν ξερω τις 10 κατηγοριες  που τα πηγαινε χειροτερα τι ηταν η κανονικη κλαση και τι προεβλεψε λαθος 'να μην δειχνω εκει που το μοντελο τα πηγε καλ…diag = np.diag(cm)
col_sums = cm.sum(axis=0); row_sums = cm.sum(axis=1)
precision = np.divide(diag, col_sums, out=np.zeros_like(diag, float), where=col_sums != 0)
recall    = np.divide(diag, row_sums, out=np.zeros_like(diag, float), where=row_sums != 0)
f1 = np.divide(2 * precision * recall, precision + recall,
               out=np.zeros_like(diag, float), where=(precision + recall) != 0)

worst = np.argsort(f1)[:20]   # 20 weakest classes
fig, ax = plt.subplots(figsize=(10, 8))
y = np.arange(len(worst))
ax.barh(y, f1[worst], color="#5DCAA5")
ax.set_yticks(y); ax.set_yticklabels([FINE_LABELS[i] for i in worst])
ax.set_xlabel("F1 score"); ax.set_title("20 worst fine classes by F1")
plt.tight_layout()
plt.savefig(run_dir / "per_class_f1_worst20.png", dpi=150)
plt.show()

# %% ===== CELL 19: load fine test task + rebuild model to inspect predictions =====
from data import load_cifar100
from data.tasks import make_cifar100_multiclass_task

test_split = load_cifar100("test")
test_task = make_cifar100_multiclass_task(test_split, label_level="fine", seed=42)
test_images = test_task.images   # (10000, 32, 32, 3) uint8
test_labels = test_task.labels   # (10000,) in [0, 100)
print(test_images.shape, test_labels.shape)

from models.efficientnet_b0 import build_efficientnet_b0
# Rebuild with the SAME hyperparams as the run_dir you picked in CELL 13.
# This example matches CONFIG B (block6 partial unfreeze) — adjust
# freeze_backbone / unfreeze_from / input_size if you picked a different run.
model = build_efficientnet_b0(
    num_classes=100,
    freeze_backbone=False,
    unfreeze_from="block6",
    freeze_bn=True,
    input_size=128,
)
model.load_weights(run_dir / "weights.h5")

# %% ===== CELL 20: predict + per-category inspection =====
test_images_normalized = test_images.astype(np.float32) / 255.0
probs = model.predict(test_images_normalized, batch_size=64, verbose=1)
preds = probs.argmax(axis=1)
confidences = probs.max(axis=1)

CATEGORY = "trout"   # <- change to any of the 100 fine class names
cat_id = FINE_LABELS.index(CATEGORY)

is_this_class = test_labels == cat_id
correct_idx = np.flatnonzero(is_this_class & (preds == cat_id))[:5]
wrong_idx   = np.flatnonzero(is_this_class & (preds != cat_id))[:5]
print(f"'{CATEGORY}': {len(correct_idx)} correct shown, {len(wrong_idx)} wrong shown "
      f"(total {is_this_class.sum()}, acc {(preds[is_this_class] == cat_id).mean():.2%})")

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, idx in enumerate(correct_idx):
    axes[0, i].imshow(test_images[idx])
    axes[0, i].set_title(f"OK {FINE_LABELS[preds[idx]]}\n({confidences[idx]:.0%})", fontsize=8, color="green")
    axes[0, i].axis("off")
for i, idx in enumerate(wrong_idx):
    axes[1, i].imshow(test_images[idx])
    axes[1, i].set_title(f"X pred: {FINE_LABELS[preds[idx]]}\n({confidences[idx]:.0%})", fontsize=8, color="red")
    axes[1, i].axis("off")
fig.suptitle(f"Category: {CATEGORY} (true label for every image)", fontsize=12)
plt.tight_layout()
plt.savefig(Path(run_dir).parent / f"examples_{CATEGORY}.png", dpi=150)
plt.show()καλο θα ηταν αν ξερω τις 10 κατηγοριες  που τα πηγαινε χειροτερα τι ηταν η κανονικη κλαση και τι προεβλεψε λαθος 'να μην δειχνω εκει που το μοντελο τα πηγε καλα'Show more1:59 μ.μ.Claude responded: Καλή σκέψη — τα 5 "OK" παραδείγματα στο CELL 20 δεν προσθέτουν τίποτα όταν θέλεις να δεις πού χάνεται το μοντέλο.Καλή σκέψη — τα 5 "OK" παραδείγματα στο CELL 20 δεν προσθέτουν τίποτα όταν θέλεις να δεις πού χάνεται το μοντέλο. Σου γράφω ένα νέο κελί που αντικαθιστά την ιδέα: παίρνει τις 10 χειρότερες κλάσεις (από το worst array που υπολόγισες ήδη στο F1 κελί), και για κάθε μία δείχνει μόνο λάθος προβλέψεις — με τη σωστή κλάση και τι προέβλεψε αντί.
python# %% ===== CELL 20 (revised): show ONLY wrong predictions for the 10 worst classes =====
test_images_normalized = test_images.astype(np.float32) / 255.0
probs = model.predict(test_images_normalized, batch_size=64, verbose=1)
preds = probs.argmax(axis=1)
confidences = probs.max(axis=1)

worst_10 = worst[:10]   # reuse the F1-sorted array from CELL 18

fig, axes = plt.subplots(10, 5, figsize=(14, 26))

for row, cat_id in enumerate(worst_10):
    category = FINE_LABELS[cat_id]
    is_this_class = test_labels == cat_id
    wrong_idx = np.flatnonzero(is_this_class & (preds != cat_id))

    n_wrong = is_this_class.sum() - (preds[is_this_class] == cat_id).sum()
    acc = (preds[is_this_class] == cat_id).mean()

    shown = wrong_idx[:5]
    for col in range(5):
        ax = axes[row, col]
        if col < len(shown):
            idx = shown[col]
            ax.imshow(test_images[idx])
            ax.set_title(
                f"true: {category}\npred: {FINE_LABELS[preds[idx]]} ({confidences[idx]:.0%})",
                fontsize=8, color="red"
            )
        ax.axis("off")
        if col == 0:
            ax.text(-0.15, 0.5, f"F1 {f1[cat_id]:.2f}\nacc {acc:.0%}\n({n_wrong} wrong)",
                     transform=ax.transAxes, fontsize=8, va="center", ha="right")

fig.suptitle("10 worst fine classes — misclassified examples only", fontsize=14, y=1.0)
plt.tight_layout()
plt.savefig(Path(run_dir).parent / "worst10_misclassified.png", dpi=150)
plt.show()

# Set run_dir for Full Unfreeze

In [ ]:
from pathlib import Path
run_dir = Path("results/efficientnet_b0_fine_ft/efficientnet_b0_fine_ft_in128_ep40")
get_ipython().system(f'ls -la {run_dir}')


# Load History and Metrics for Full Unfreeze

In [ ]:
import json

with open(run_dir / "history.json") as f:
    history = json.load(f)
with open(run_dir / "metrics.json") as f:
    metrics = json.load(f)

print("History keys:", list(history.keys()))
print("=== Final test metrics (fine / 100 classes, FULL UNFREEZE) ===")
print(f"Accuracy:        {metrics['accuracy']:.4f}")
if "top_3_accuracy" in metrics:
    print(f"Top-3 accuracy:  {metrics['top_3_accuracy']:.4f}")
if "top_5_accuracy" in metrics:
    print(f"Top-5 accuracy:  {metrics['top_5_accuracy']:.4f}")
print(f"Macro precision: {metrics['macro_precision']:.4f}")
print(f"Macro recall:    {metrics['macro_recall']:.4f}")
print(f"Macro F1:        {metrics['macro_f1']:.4f}")
print(f"Weighted F1:     {metrics['weighted_f1']:.4f}")

# Append Metrics to Summary CSV for Full Unfreeze

In [ ]:
import csv

summary_path = Path("results/summary.csv")
COLUMNS = ["task_name", "run_name", "task_type", "label_level", "target_labels",
           "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "threshold",
           "top_3_accuracy", "top_5_accuracy", "macro_f1", "weighted_f1"]

row = {c: "" for c in COLUMNS}
row.update({
    "task_name": run_dir.parent.name,
    "run_name": run_dir.name,
    "task_type": "multiclass",
    "label_level": "fine",
    "accuracy": metrics.get("accuracy", ""),
    "top_3_accuracy": metrics.get("top_3_accuracy", ""),
    "top_5_accuracy": metrics.get("top_5_accuracy", ""),
    "macro_f1": metrics.get("macro_f1", ""),
    "weighted_f1": metrics.get("weighted_f1", ""),
})

write_header = not summary_path.exists()
with summary_path.open("a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=COLUMNS)
    if write_header:
        w.writeheader()
    w.writerow(row)
print(f"Appended {row['run_name']} to {summary_path}")

# Plot Training Curves for Full Unfreeze

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(history["accuracy"], label="train")
axes[1].plot(history["val_accuracy"], label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout()
plt.savefig(run_dir / "training_curves.png", dpi=150)
plt.show()

# Define Fine-Grained Labels (Redundant)

In [ ]:
FINE_LABELS = [
    "apple", "aquarium_fish", "baby", "bear", "beaver", "bed", "bee", "beetle",
    "bicycle", "bottle", "bowl", "boy", "bridge", "bus", "butterfly", "camel",
    "can", "castle", "caterpillar", "cattle", "chair", "chimpanzee", "clock",
    "cloud", "cockroach", "couch", "crab", "crocodile", "cup", "dinosaur",
    "dolphin", "elephant", "flatfish", "forest", "fox", "girl", "hamster",
    "house", "kangaroo", "keyboard", "lamp", "lawn_mower", "leopard", "lion",
    "lizard", "lobster", "man", "maple_tree", "motorcycle", "mountain", "mouse",
    "mushroom", "oak_tree", "orange", "orchid", "otter", "palm_tree", "pear",
    "pickup_truck", "pine_tree", "plain", "plate", "poppy", "porcupine",
    "possum", "rabbit", "raccoon", "ray", "road", "rocket", "rose", "sea",
    "seal", "shark", "shrew", "skunk", "skyscraper", "snail", "snake", "spider",
    "squirrel", "streetcar", "sunflower", "sweet_pepper", "table", "tank",
    "telephone", "television", "tiger", "tractor", "train", "trout", "tulip",
    "turtle", "wardrobe", "whale", "willow_tree", "wolf", "woman", "worm",
]
assert len(FINE_LABELS) == 100

# Plot Confusion Matrix for Full Unfreeze

In [ ]:
import numpy as np

cm = np.array(metrics["confusion_matrix"])
fig, ax = plt.subplots(figsize=(11, 10))
im = ax.imshow(cm, cmap="Blues")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion matrix — fine (100 classes, full unfreeze)")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(run_dir / "confusion_matrix.png", dpi=150)
plt.show()

# Load Data and Rebuild Model for Full Unfreeze Predictions

In [ ]:
from data import load_cifar100
from data.tasks import make_cifar100_multiclass_task

test_split = load_cifar100("test")
test_task = make_cifar100_multiclass_task(test_split, label_level="fine", seed=42)
test_images = test_task.images
test_labels = test_task.labels
print(test_images.shape, test_labels.shape)

from models.efficientnet_b0 import build_efficientnet_b0
# Must match CONFIG C exactly: freeze_backbone=False, unfreeze_from="all", freeze_bn=True
model = build_efficientnet_b0(
    num_classes=100,
    freeze_backbone=False,
    unfreeze_from="all",
    freeze_bn=True,
    input_size=128,
)
model.load_weights(run_dir / "weights.h5")


# Plot Worst-10 F1 for Full Unfreeze

In [ ]:
# %% ===== Worst-10 F1 plot — FULL UNFREEZE =====
run_dir_ft = Path("results/efficientnet_b0_fine_ft/efficientnet_b0_fine_ft_in128_ep40")

with open(run_dir_ft / "metrics.json") as f:
    metrics_ft = json.load(f)

cm_ft = np.array(metrics_ft["confusion_matrix"])

FINE_LABELS = [
    "apple", "aquarium_fish", "baby", "bear", "beaver", "bed", "bee", "beetle",
    "bicycle", "bottle", "bowl", "boy", "bridge", "bus", "butterfly", "camel",
    "can", "castle", "caterpillar", "cattle", "chair", "chimpanzee", "clock",
    "cloud", "cockroach", "couch", "crab", "crocodile", "cup", "dinosaur",
    "dolphin", "elephant", "flatfish", "forest", "fox", "girl", "hamster",
    "house", "kangaroo", "keyboard", "lamp", "lawn_mower", "leopard", "lion",
    "lizard", "lobster", "man", "maple_tree", "motorcycle", "mountain", "mouse",
    "mushroom", "oak_tree", "orange", "orchid", "otter", "palm_tree", "pear",
    "pickup_truck", "pine_tree", "plain", "plate", "poppy", "porcupine",
    "possum", "rabbit", "raccoon", "ray", "road", "rocket", "rose", "sea",
    "seal", "shark", "shrew", "skunk", "skyscraper", "snail", "snake", "spider",
    "squirrel", "streetcar", "sunflower", "sweet_pepper", "table", "tank",
    "telephone", "television", "tiger", "tractor", "train", "trout", "tulip",
    "turtle", "wardrobe", "whale", "willow_tree", "wolf", "woman", "worm",
]

diag = np.diag(cm_ft)
col_sums = cm_ft.sum(axis=0); row_sums = cm_ft.sum(axis=1)
precision_ft = np.divide(diag, col_sums, out=np.zeros_like(diag, float), where=col_sums != 0)
recall_ft    = np.divide(diag, row_sums, out=np.zeros_like(diag, float), where=row_sums != 0)
f1_ft = np.divide(2 * precision_ft * recall_ft, precision_ft + recall_ft,
                   out=np.zeros_like(diag, float), where=(precision_ft + recall_ft) != 0)

worst10_ft = np.argsort(f1_ft)[:10]   # 10 weakest classes for the full-unfreeze run

fig, ax = plt.subplots(figsize=(10, 6))
y = np.arange(len(worst10_ft))
ax.barh(y, f1_ft[worst10_ft], color="#E07A5F")
ax.set_yticks(y); ax.set_yticklabels([FINE_LABELS[i] for i in worst10_ft])
ax.set_xlabel("F1 score"); ax.set_title("10 worst fine classes by F1 — full unfreeze")
plt.tight_layout()
plt.savefig(run_dir_ft / "per_class_f1_worst10.png", dpi=150)
plt.show()

# Plot Worst-10 Misclassified Examples Grid for Full Unfreeze

In [ ]:
# %% ===== Worst-10 misclassified examples grid — FULL UNFREEZE =====
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

run_dir_ft = Path("results/efficientnet_b0_fine_ft/efficientnet_b0_fine_ft_in128_ep40")

with open(run_dir_ft / "metrics.json") as f:
    metrics_ft = json.load(f)

cm_ft = np.array(metrics_ft["confusion_matrix"])

diag = np.diag(cm_ft)
col_sums = cm_ft.sum(axis=0); row_sums = cm_ft.sum(axis=1)
precision_ft = np.divide(diag, col_sums, out=np.zeros_like(diag, float), where=col_sums != 0)
recall_ft    = np.divide(diag, row_sums, out=np.zeros_like(diag, float), where=row_sums != 0)
f1_ft = np.divide(2 * precision_ft * recall_ft, precision_ft + recall_ft,
                   out=np.zeros_like(diag, float), where=(precision_ft + recall_ft) != 0)

worst_ft = np.argsort(f1_ft)[:10]

# Rebuild the full-unfreeze model and predict on the test set
from data import load_cifar100
from data.tasks import make_cifar100_multiclass_task
from models.efficientnet_b0 import build_efficientnet_b0

test_split = load_cifar100("test")
test_task = make_cifar100_multiclass_task(test_split, label_level="fine", seed=42)
test_images = test_task.images
test_labels = test_task.labels

model_ft = build_efficientnet_b0(
    num_classes=100,
    freeze_backbone=False,
    unfreeze_from="all",
    freeze_bn=True,
    input_size=128,
)
model_ft.load_weights(run_dir_ft / "weights.h5")

test_images_normalized = test_images.astype(np.float32) / 255.0
probs_ft = model_ft.predict(test_images_normalized, batch_size=64, verbose=1)
preds_ft = probs_ft.argmax(axis=1)
confidences_ft = probs_ft.max(axis=1)

# Build the grid
fig, axes = plt.subplots(10, 5, figsize=(14, 26))

for row, cat_id in enumerate(worst_ft):
    category = FINE_LABELS[cat_id]
    is_this_class = test_labels == cat_id
    wrong_idx = np.flatnonzero(is_this_class & (preds_ft != cat_id))

    acc = (preds_ft[is_this_class] == cat_id).mean()
    n_wrong = len(wrong_idx)

    shown = wrong_idx[:5]
    for col in range(5):
        ax = axes[row, col]
        if col < len(shown):
            idx = shown[col]
            ax.imshow(test_images[idx])
            ax.set_title(
                f"true: {category}\npred: {FINE_LABELS[preds_ft[idx]]} ({confidences_ft[idx]:.0%})",
                fontsize=8, color="red"
            )
        ax.axis("off")
        if col == 0:
            ax.text(-0.15, 0.5, f"F1 {f1_ft[cat_id]:.2f}\nacc {acc:.0%}\n({n_wrong} wrong)",
                     transform=ax.transAxes, fontsize=8, va="center", ha="right")

fig.suptitle("10 worst fine classes — misclassified examples only (full unfreeze)", fontsize=14, y=1.0)
plt.tight_layout()
plt.savefig(run_dir_ft / "worst10_misclassified.png", dpi=150)
plt.show()

In [ ]:
# %% Βεβαιώσου ότι είσαι στο σωστό branch
get_ipython().system('git branch')
get_ipython().system('git status')

In [ ]:
get_ipython().system('pwd')
get_ipython().system('ls')

In [ ]:
# %% Clone the repo
get_ipython().system('git clone https://github.com/Fgram-devAI/deepl-cifar100-image-analysis.git')

In [ ]:
# %% cd into it (no # in front of %cd!)
%cd /content/deepl-cifar100-image-analysis

In [ ]:
# %% Checkout the existing branch (no -b, it already exists remotely)
get_ipython().system('git checkout experiment/efficientnet-b0-fine')
get_ipython().system('git status')

In [ ]:
# %% Καθάρισε τα outputs ΑΠΟ ΤΟ ΙΔΙΟ ΤΟ ΖΩΝΤΑΝΟ NOTEBOOK και αποθήκευσέ το ως νέο αρχείο
import json

# Colab εκθέτει το δικό του notebook metadata μέσω αυτής της global μεταβλητής
from google.colab import _message
notebook_json = _message.blocking_request('get_ipynb', timeout_sec=30)

nb = notebook_json['ipynb']

for cell in nb['cells']:
    if cell.get('cell_type') == 'code':
        cell['outputs'] = []
        cell['execution_count'] = None

with open('b0_fine_100classes_clean.ipynb', 'w') as f:
    json.dump(nb, f, indent=1)

print("Καθαρισμένο notebook αποθηκεύτηκε: b0_fine_100classes_clean.ipynb")
import os
print("Μέγεθος:", os.path.getsize('b0_fine_100classes_clean.ipynb'), "bytes")